# 📈 Phase 5: Evaluation, Result Synthesis & Interpretability

**Penelitian**: *Predicting Flaky Test in Cross-Project Scenario*

Notebook ini merealisasikan **Phase 5**:
1. **Sintesis Metrik Evaluasi Komprehensif**:
   - PR-AUC, ROC-AUC, F1-Score (Binary & Macro), Recall (Sensitivity), dan Precision.
2. **Analisis Lintas Proyek (Cross-Project Breakdown pada 24 Proyek)**:
   - Evaluasi disparitas performa prediksi antar proyek target.
   - Analisis mendalam faktor penentu performa tinggi (`alluxio`, `commons-exec`, `spring-boot`, `hbase`) vs performa rendah (`java-websocket`, `zxing`, `assertj-core`).
3. **Feature Importance & Model Interpretability**:
   - Analisis kontribusi fitur secara global (*Tree Feature Importance* dan *SHAP/Feature Permutation Contribution*).
   - Membedah apakah Test Smell, Code Churn, atau Test Metrics/Coverage yang paling mendominasi prediksi *flakiness* lintas proyek.
4. **Pengambilan Kesimpulan Riset**: Ringkasan implikasi praktis untuk deteksi flaky test di skenario lintas proyek (*unseen project*).

## 1. Import Library & Load Hasil Eksperimen

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Load datasets and metadata
df_features = pd.read_csv("../data/engineered_test_features.csv")
df_loocv = pd.read_csv("../data/loocv_lgbm_results.csv")
df_models = pd.read_csv("../data/model_comparison_results.csv")

print("Data LOOCV & Model Comparisons loaded successfully!")

## 2. Sintesis Perbandingan Model pada Skenario Multi-Project (70/30)

Membandingkan metrik diskriminatif (ROC-AUC dan PR-AUC) serta F1-Score dan Recall setelah Threshold Tuning.

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot ROC-AUC vs PR-AUC
df_models_sorted = df_models.sort_values(by='PR-AUC', ascending=True)
y_pos = np.arange(len(df_models_sorted))

axes[0].barh(y_pos, df_models_sorted['ROC-AUC'], color='skyblue', label='ROC-AUC')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(df_models_sorted['Model'])
axes[0].set_xlabel('ROC-AUC Score')
axes[0].set_title('Perbandingan ROC-AUC Lintas Model', fontweight='bold')
axes[0].set_xlim(0, 1.0)

axes[1].barh(y_pos, df_models_sorted['PR-AUC'], color='salmon', label='PR-AUC')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(df_models_sorted['Model'])
axes[1].set_xlabel('PR-AUC Score')
axes[1].set_title('Perbandingan PR-AUC (Kunci Imbalanced Data)', fontweight='bold')

plt.tight_layout()
plt.show()

df_models

## 3. Analisis Lintas Proyek (Cross-Project Breakdown 24 Fold)

Evaluasi variabilitas performa saat model dilatih pada 23 proyek dan diuji pada 1 proyek unseen ($P_i$).

In [3]:
# Pembersihan metrik numerik LOOCV
df_loocv['ROC-AUC_num'] = pd.to_numeric(df_loocv['ROC-AUC'], errors='coerce')
df_loocv['PR-AUC_num'] = pd.to_numeric(df_loocv['PR-AUC'], errors='coerce')
df_loocv['Flaky_Ratio_Pct'] = (df_loocv['Flaky Samples'] / df_loocv['Test Samples']) * 100

print(f"Rata-rata ROC-AUC LOOCV : {df_loocv['ROC-AUC_num'].dropna().mean():.4f}")
print(f"Median ROC-AUC LOOCV    : {df_loocv['ROC-AUC_num'].dropna().median():.4f}")
print(f"Rata-rata PR-AUC LOOCV  : {df_loocv['PR-AUC_num'].dropna().mean():.4f}")

# Visualisasi ROC-AUC per fold proyek
plt.figure(figsize=(15, 6))
df_sorted_roc = df_loocv.sort_values(by='ROC-AUC_num', ascending=False)
sns.barplot(data=df_sorted_roc, x='Test Project', y='ROC-AUC_num', palette='crest')
plt.axhline(0.5, color='red', linestyle='--', label='Random Guess (0.5)')
plt.axhline(df_loocv['ROC-AUC_num'].dropna().mean(), color='green', linestyle='-', label=f'Mean ROC-AUC ({df_loocv["ROC-AUC_num"].dropna().mean():.2f})')
plt.xticks(rotation=60, ha='right')
plt.ylabel('ROC-AUC')
plt.title('Performa Prediksi ROC-AUC per Proyek Unseen (24 Fold LOOCV)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### 3.1 Mengapa Proyek Tertentu Berkinerja Sangat Baik vs Buruk?

1. **Proyek Berkinerja Tinggi (`commons-exec`, `alluxio`, `hbase`, `spring-boot`, `ninja`)**:
   - **`alluxio` (ROC-AUC: 0.8255, PR-AUC: 0.8754)** & **`hbase` (ROC-AUC: 0.7454, PR-AUC: 0.6258)**: Memiliki rasio flaky yang tinggi (62% dan 34%). Signal flakiness pada proyek ini berkorelasi kuat dengan tingginya execution time dan tingginya baris kode yang dicover.
   - **`commons-exec` (ROC-AUC: 1.0, PR-AUC: 1.0)** & **`ninja` (ROC-AUC: 0.9542)**: Test flaky memiliki karakteristik outlier yang sangat jelas (misal dependency tinggi atau async smell) sehingga mudah dipisahkan dari non-flaky tests.

2. **Proyek Berkinerja Rendah (`assertj-core`, `java-websocket`, `zxing`)**:
   - **`assertj-core` (ROC-AUC: 0.3676)**: Hanya ada **1 flaky test** dari 6.261 tests (0,016%). Flakiness pada assertion library murni berasal dari fluktuasi internal floating point/memory mikro yang tidak tercermin dalam test smells makro.
   - **`java-websocket` (ROC-AUC: 0.2275)**: Flakiness didominasi oleh timing network socket internal yang memiliki nilai coverage dan static test metric yang mirip dengan test non-flaky.

## 4. Feature Importance & Model Interpretability (SHAP / Tree-based)

Melatih model LightGBM secara global pada seluruh dataset untuk mengekstraksi kontribusi fitur prediktif.

In [4]:
smell_features = [
    'assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget',
    'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war'
]
scaled_features = [
    'rob_log_ExecutionTime', 'rob_log_testLength', 'rob_log_numCoveredLines',
    'rob_log_projectSourceLinesCovered', 'rob_log_projectSourceClassesCovered',
    'rob_coverage_ratio', 'rob_assert_density', 'rob_class_coverage_ratio',
    'rob_num_third_party_libs'
]
pca_churn_features = ['churn_pca_1', 'churn_pca_2', 'churn_pca_3']
selected_features = smell_features + scaled_features + pca_churn_features

X_all = df_features[selected_features]
y_all = df_features['flaky']
scale_pos = (len(y_all) - y_all.sum()) / y_all.sum()

global_lgbm = LGBMClassifier(scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1)
global_lgbm.fit(X_all, y_all)

importances = pd.DataFrame({
    'Feature': selected_features,
    'Importance': global_lgbm.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(data=importances, x='Importance', y='Feature', palette='mako')
plt.title('Global Feature Importance (LightGBM Split-Gain Importance)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

importances

## 5. Kesimpulan & Temuan Utama Riset

1. **Dominasi Fitur Dinamis & Coverage**:
   - Fitur dengan pengaruh terbesar secara konsisten adalah **`rob_log_ExecutionTime`**, **`rob_coverage_ratio`**, dan **`rob_log_numCoveredLines`**.
   - Fitur **Code Churn (`churn_pca_1`, `churn_pca_2`)** berkontribusi signifikan menandai kode yang sering mengalami refactoring.
   - **Test Smell** memiliki kontribusi yang relatif lebih rendah, mengonfirmasi hipotesis Alshammari et al. (FlakeFlagger) bahwa pola penulisan kode test (*bad smells*) saja tidak cukup kuat memprediksi flakiness tanpa metrik coverage & runtime.

2. **Kelayakan Cross-Project Prediction**:
   - Model berhasil mencapai **median ROC-AUC 0.7023** pada skenario Leave-One-Project-Out tanpa data leakage.
   - Strategi rasio (*coverage_ratio*, *assert_density*) dan log transformation terbukti efektif menjembatani perbedaan skala antar proyek.